In [40]:
import pandas as pd
import numpy as np
import re
import json
import ast
from pathlib import Path
import requests
import time
from tqdm import tqdm

EXTRACT_DIR = Path.cwd().parent / "Extract"
print('Cartella dati:', EXTRACT_DIR)

Cartella dati: c:\Users\Dario\OneDrive\Desktop\Project II\Extract


In [41]:
PROV_SIGLA = {
    'Varese': 'VA', 'Como': 'CO', 'Sondrio': 'SO', 'Milano': 'MI',
    'Bergamo': 'BG', 'Brescia': 'BS', 'Pavia': 'PV', 'Cremona': 'CR',
    'Mantova': 'MN', 'Lecco': 'LC', 'Lodi': 'LO',
    'Monza e della Brianza': 'MB',
}

df_res = pd.read_csv(
    'residenti_lombardia_per_fascia.csv',
    sep=';', encoding='utf-8-sig'
)

comune_to_sigla = (
    df_res[['Comune', 'Provincia']]
    .drop_duplicates('Comune')
    .assign(sigla=lambda d: d['Provincia'].map(PROV_SIGLA))
    .set_index('Comune')['sigla']
    .to_dict()
)

def lookup_sigla(comune: str) -> str:
    return comune_to_sigla.get(comune)

print(f'Comuni mappati: {len(comune_to_sigla)}')

Comuni mappati: 1502


In [42]:
def split_ASST(nome: str) -> tuple[str, str]:
    
    m = re.search(r'\s*-\s*(ASST\s+.+)$', nome, re.IGNORECASE)
    if m:
        return nome[:m.start()].strip(), m.group(1).strip()
    return nome.strip(), ''


def normalizza_indirizzo_micuro(raw: str) -> str:
    if pd.isna(raw):
        return raw
    raw = ' '.join(raw.split())
    m = re.match(r'^(.+?)\s*-\s*\d{5}\s+(.+?)\s*\(([A-Z]{2})\)$', raw)
    if m:
        return f'{m.group(1).strip().upper()} - {m.group(2).strip().upper()} ({m.group(3).upper()})'
    return raw.upper()


def normalizza_indirizzo_cdc(raw: str) -> str:
    if pd.isna(raw):
        return raw

    raw = ' '.join(raw.split())

    m = re.match(r'^(.+)\s+-\s+(.+\([A-Z]{2}\))$', raw)

    via_part = m.group(1).strip()
    city_part = m.group(2).strip()

    via_m = re.match(r'^(.+?)\s*,?\s+(\d[\w/]*)$', via_part)
    if via_m:
        via_part = f'{via_m.group(1).strip()}, {via_m.group(2).strip()}'

    return f'{via_part.upper()} - {city_part.upper()}'


# Test split_ASST
casi = [
    'Ospedale Carlo Ondoli di Angera - ASST Sette Laghi',
    'Carlo Erba - Renaldi',
    'Renaldi di Menaggio - ASST Lariana',
    "Sant'Ambrogio di Milano - Gruppo San Donato",
    'Istituto Europeo di Oncologia di Milano',
]
for c in casi:
    print(split_ASST(c))

('Ospedale Carlo Ondoli di Angera', 'ASST Sette Laghi')
('Carlo Erba - Renaldi', '')
('Renaldi di Menaggio', 'ASST Lariana')
("Sant'Ambrogio di Milano - Gruppo San Donato", '')
('Istituto Europeo di Oncologia di Milano', '')


In [43]:
df_micuro = pd.read_csv(
    EXTRACT_DIR / 'micuro_lombardia_strutture.csv',
    encoding='utf-8-sig'
)

df_micuro['nome'], df_micuro['ASST'] = zip(
    *df_micuro['nome'].apply(split_ASST)
)

df_micuro['nome'] = df_micuro['nome'].str.upper()
df_micuro['ASST'] = df_micuro['ASST'].str.upper()

for col in ['n_valutazioni', 'n_aree']:
    df_micuro[col] = pd.to_numeric(df_micuro[col], errors='coerce').fillna(0).astype(int)

df_micuro['indirizzo'] = df_micuro['indirizzo'].apply(normalizza_indirizzo_micuro)

df_micuro['aree_specialistiche'] = df_micuro['aree_specialistiche'].apply(
    lambda x: [a.strip().upper() for a in x.split(',')] if isinstance(x, str) else []
)

df_micuro = df_micuro.drop(columns=['href'], errors='ignore')

print(f'micuro: {len(df_micuro)} righe')
df_micuro[['nome', 'ASST', 'indirizzo', 'n_valutazioni', 'aree_specialistiche']].head(5)

micuro: 134 righe


,nome,ASST,indirizzo,n_valutazioni,aree_specialistiche
0,OSPEDALE CARLO ONDOLI DI ANGERA,ASST SETTE LAGHI,"VIA BORDINI, 9 - ANGERA (VA)",3,"[ANALISI DI LABORATORIO, CARDIOLOGIA, CHIRURGI..."
1,OSPEDALE DEI BAMBINI DI BRESCIA,ASST SPEDALI CIVILI,"VIA DEL MEDOLO, 2 - BRESCIA (BS)",2,"[ANALISI DI LABORATORIO, CARDIOLOGIA, CHIRURGI..."
2,OSPEDALE FILIPPO DEL PONTE DI VARESE,ASST SETTE LAGHI,"VIA FILIPPO DEL PONTE, 19 - VARESE (VA)",2,"[CARDIOLOGIA, CHIRURGIA GENERALE, FISIOPATOLOG..."
3,IRCCS MAUGERI LUMEZZANE,,"VIA GIUSEPPE MAZZINI, 129 - LUMEZZANE (BS)",1,"[ALLERGOLOGIA, ANALISI DI LABORATORIO, ANGIOLO..."
4,OSPEDALE DI SEREGNO,ASST BRIANZA,"VIA VERDI, 2 - SEREGNO (MB)",1,"[DIETOLOGIA E NUTRIZIONE CLINICA, ENDOCRINOLOG..."


In [44]:
df_cdc = pd.read_csv(
    EXTRACT_DIR / "case_di_comunita.csv",
    sep=';', encoding='utf-8-sig'
)

df_cdc['INDIRIZZO STRUTTURA'] = df_cdc.apply(
    lambda r: normalizza_indirizzo_cdc(r['INDIRIZZO STRUTTURA']),
    axis=1
)

df_cdc['% SERVIZI ATTIVI'] = (
    df_cdc['% SERVIZI ATTIVI']
    .str.replace('%', '', regex=False)
    .astype(float)
)
df_cdc = df_cdc[df_cdc['% SERVIZI ATTIVI'] > 0].reset_index(drop=True)
df_cdc = df_cdc.drop(columns=['% SERVIZI ATTIVI', 'ATS', 'NUMERO SERVIZI ATTIVI', 'COMUNE'],
                     errors='ignore')

df_cdc['ASST'] = df_cdc['ASST'].str.upper()
df_cdc['CDC']  = df_cdc['CDC'].apply(lambda n: n.replace("a'","à").upper())

df_cdc[['CDC', 'ASST', 'INDIRIZZO STRUTTURA']].head(5)

,CDC,ASST,INDIRIZZO STRUTTURA
0,CASA DI COMUNITÀ GIUDITTA VIMERCATE,ASST DELLA BRIANZA,"VIA GIUDITTA BRAMBILLA, 11 - VIMERCATE (MB)"
1,CASA DI COMUNITÀ BRESCIA NAVE,ASST DEGLI SPEDALI CIVILI DI BRESCIA,"VIA BRESCIA, 155 - NAVE (BS)"
2,CASA DI COMUNITÀ AGOI BORMIO,ASST DELLA VALTELLINA E DELL'ALTO LARIO,"VIA AGOI, 8 - BORMIO (SO)"
3,CASA DI COMUNITÀ RAFFAELLO TRAVAGLIATO,ASST DEGLI SPEDALI CIVILI DI BRESCIA,"VIA RAFFAELLO, 24 - TRAVAGLIATO (BS)"
4,CASA DI COMUNITÀ DON ORIONE MILANO,ASST FATEBENEFRATELLI SACCO,"VIA DON ORIONE, 2 - MILANO (MI)"


In [45]:
micuro_merge = df_micuro.rename(columns={
    'nome': 'NOME_STRUTTURA',
    'indirizzo': 'INDIRIZZO',
})

cdc_merge = df_cdc.rename(columns={
    'CDC': 'NOME_STRUTTURA',
    'INDIRIZZO STRUTTURA': 'INDIRIZZO',
})

df_merged = pd.concat([micuro_merge, cdc_merge], ignore_index=True, sort=False)

df_merged['ASST'] = df_merged['ASST'].fillna('')

cols_testa = ['NOME_STRUTTURA', 'ASST', 'INDIRIZZO']
altri = [c for c in df_merged.columns if c not in cols_testa]
df_merged = df_merged[cols_testa + altri]

df_merged[['NOME_STRUTTURA', 'ASST', 'INDIRIZZO']].sample(5)

,NOME_STRUTTURA,ASST,INDIRIZZO
76,OSPEDALE DI GALLARATE,ASST VALLE OLONA,"VIA EUSEBIO PASTORI, 4 - GALLARATE (VA)"
163,CASA DI COMUNITÀ CERERIA CHIAVENNA,ASST DELLA VALTELLINA E DELL'ALTO LARIO,"VIA CERERIA, 4 - CHIAVENNA (SO)"
212,CASA DI COMUNITÀ DON MOLETTA VAPRIO D'ADDA,ASST MELEGNANO E DELLA MARTESANA,"VIA DON MOLETTA, 22 - VAPRIO D'ADDA (MI)"
280,CASA DI COMUNITÀ FACCANONI SARNICO,ASST DI BERGAMO EST,"VIA FACCANONI, 6 - SARNICO (BG)"
304,CASA DI COMUNITÀ ZANICA,ASST DI BERGAMO OVEST,"VIA SERIO, 1/A - ZANICA (BG)"


In [46]:
mask = df_merged["NOME_STRUTTURA"] == "OSPEDALE OGLIO PO DI CASALMAGGIORE"
df_merged.loc[mask, "INDIRIZZO"] = "VIA STAFFOLO, 51 - CASALMAGGIORE (CR)"

mask = df_merged["NOME_STRUTTURA"] == "OSPEDALE M. O. A. LOCATELLI DI PIARIO"
df_merged.loc[mask, "INDIRIZZO"] = "VIA GROPPINO, 22 - PIARIO (BG)"

mask = df_merged["NOME_STRUTTURA"] == "ICS MAUGERI LISSONE"
df_merged.loc[mask, "INDIRIZZO"] = "Via Monsignore Bernasconi, 16 - Lissone (MB)".upper()

mask = df_merged["NOME_STRUTTURA"] == "IRCCS MAUGERI CASTEL GOFFREDO"
df_merged.loc[mask, "INDIRIZZO"] = "Via Ospedale, 36 -  Castel Goffredo (MN)".upper()

mask = df_merged["NOME_STRUTTURA"] == "IRCCS MAUGERI MONTESCANO"
df_merged.loc[mask, "INDIRIZZO"] = "Via per Montescano, 35 - Montescano (PV)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ BETELLI DALMINE"
df_merged.loc[mask, "INDIRIZZO"] = "Viale Natale Betelli, 2 - Dalmine (BG)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ OSPEDALE CALCINATE"
df_merged.loc[mask, "INDIRIZZO"] = "Piazza Ospedale, 3 - Calcinate (BG)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ QUINTINI DI VONA CASSANO D'ADDA"
df_merged.loc[mask, "INDIRIZZO"] = "Via Q. di Vona, 41 - Cassano d'Adda (MI)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ DEL LAZZARETTO BELLAGIO"
df_merged.loc[mask, "INDIRIZZO"] = "Via Lazzaretto, 12 - Bellagio (CO)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ CARDINAL DELL'ACQUA SESTO CALENDE"
df_merged.loc[mask, "INDIRIZZO"] = "Largo Cardinale dell'Acqua, 1 - Sesto Calende (VA)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ VANONCINI SANT'OMOBONO TERME"
df_merged.loc[mask, "INDIRIZZO"] = "Via Gianantonio Vanoncini, 25 -  Sant'omobono terme (BG)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ CORTE DEI FRATI BELLUSCO"
df_merged.loc[mask, "INDIRIZZO"] = "VIA CORTE DEI FRATI, 1 - BELLUSCO (MB)"

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ MONTEREGGIO CASATENOVO"
df_merged.loc[mask, "INDIRIZZO"] = "Via Monteregio, 15 - Casatenovo (LC)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ BERGAMO CALOLZIOCORTE"
df_merged.loc[mask, "INDIRIZZO"] = "Via Bergamo, 1 - Calolziocorte (LC)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ MANDIC MERATE"
df_merged.loc[mask, "INDIRIZZO"] = "Largo Beato Leopoldo Mandic, 1 - Merate (LC)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ LUIGI CADORNA SUZZARA"
df_merged.loc[mask, "INDIRIZZO"] = "VIA LUIGI CADORNA, 2 - SUZZARA (MN)"

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ SACCHI CASTIGLIONE DELLE STIVIERE"
df_merged.loc[mask, "INDIRIZZO"] = "Via Sacchi, 10/12 - Castiglione delle Stiviere (MN)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ DONATORI DEL SANGUE LENO"
df_merged.loc[mask, "INDIRIZZO"] = "Piazza Donatori di Sangue, 1 - Leno (BS)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ MONTEROSA VARESE"
df_merged.loc[mask, "INDIRIZZO"] = "Viale Monte Rosa, 28 - Varese (VA)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ DON MINZONI CASTELLANZA"
df_merged.loc[mask, "INDIRIZZO"] = "Viale Don G. Minzoni, 25 - Castellanza (VA)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ IV NOVEMBRE PALAZZOLO SULL'OGLIO"
df_merged.loc[mask, "INDIRIZZO"] = "Viale IV Novembre, 5 - Palazzolo sull'Oglio (BS)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ GRAMSCI FAGNANO OLONA"
df_merged.loc[mask, "INDIRIZZO"] = "Piazza Antonio Gramsci, 1 - Fagnano Olona (VA)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ BARBOLINI DARFO BOARIO TERME"
df_merged.loc[mask, "INDIRIZZO"] = "Via Barbolini Armando, 1 - Darfo Boario Terme (BS)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ MONTEGRAPPA VIGEVANO"
df_merged.loc[mask, "INDIRIZZO"] = "Viale Montegrappa, 5 - Vigevano (PV)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ CORTEOLONA-GENZONE"
df_merged.loc[mask, "INDIRIZZO"] = "Via Longobardi, 3 -  Corteolona e Genzone (PV)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ DONATORI EDOLO"
df_merged.loc[mask, "INDIRIZZO"] = "Piazza Donatori di Sangue, 1 - Edolo (BS)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ GIUSEPPE VERDI PONTE LAMBRO"
df_merged.loc[mask, "INDIRIZZO"] = "Via Giuseppe Verdi, 3 - Ponte Lambro (CO)".upper()

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ GEN. REVERBERI VESTONE"
df_merged.loc[mask, "INDIRIZZO"] = "Via Generale L. Reverberi, 2 - Vestone (BS)".upper()

In [47]:
df_merged["TIPO"] = np.where(df_merged["NOME_STRUTTURA"].str.startswith("CASA DI"),"casa di comunità".upper(),"OSPEDALE")

HUB_LIST = ["OSPEDALE PAPA GIOVANNI XXIII DI BERGAMO", "PRESIDIO OSPEDALIERO SPEDALI CIVILI DI BRESCIA",
            "PRESIDIO OSPEDALIERO ALESSANDRO MANZONI DI LECCO", "OSPEDALE CARLO POMA DI MANTOVA", "OSPEDALE NIGUARDA DI MILANO",
            "OSPEDALE FATEBENEFRATELLI E OFTALMICO DI MILANO", "OSPEDALE LUIGI SACCO DI MILANO",
            "FONDAZIONE IRCCS CA' GRANDA OSPEDALE MAGGIORE POLICLINICO DI MILANO", "OSPEDALE SAN PAOLO DI MILANO",
            "OSPEDALE CIVILE DI LEGNANO", "FONDAZIONE IRCCS SAN GERARDO DEI TINTORI DI MONZA - ATS BRIANZA", 
            "OSPEDALE POLICLINICO SAN MATTEO DI PAVIA", "OSPEDALE DI SONDRIO", "OSPEDALE DI CIRCOLO E FONDAZIONE MACCHI DI VARESE",
            "FONDAZIONE IRCCS ISTITUTO NEUROLOGICO CARLO BESTA DI MILANO", "ISTITUTO NAZIONALE DEI TUMORI",
            "IRCCS OSPEDALE SAN RAFFAELE DI MILANO - GRUPPO SAN DONATO", "HUMANITAS RESEARCH HOSPITAL DI ROZZANO",
            "IRCCS OSPEDALE GALEAZZI - SANT'AMBROGIO DI MILANO - GRUPPO SAN DONATO", "IEO - ISTITUTO EUROPEO DI ONCOLOGIA DI MILANO",
            "CENTRO CARDIOLOGICO MONZINO DI MILANO"]

conditions = [
    (df_merged["TIPO"] == "OSPEDALE") & (df_merged["NOME_STRUTTURA"].isin(HUB_LIST)),
    (df_merged["TIPO"] == "OSPEDALE") & (~df_merged["NOME_STRUTTURA"].isin(HUB_LIST)),
]
choices = ["HUB", "SPOKE"]

df_merged["CLASSIFICAZIONE"] = np.select(conditions, choices, default="casa di comunità".upper())

df_merged[["COMUNE", "PROVINCIA"]] = df_merged["INDIRIZZO"].str.extract(r'-\s+([^(]+?)\s+\(([^)]+)\)')

df_merged.rename(columns=str.upper, inplace=True)

df_merged.sample(5)

,NOME_STRUTTURA,ASST,INDIRIZZO,RATING_GLOBALE,N_VALUTAZIONI,N_AREE,AREE_SPECIALISTICHE,RATING_DIMENSIONI_JSON,RATING_PER_AREA_JSON,TIPO,CLASSIFICAZIONE,COMUNE,PROVINCIA
311,CASA DI COMUNITÀ ALBERTONI MANTOVA,ASST DI MANTOVA,"VIALE ALBERTONI, 1 - MANTOVA (MN)",NaN,NaN,NaN,NaN,NaN,NaN,CASA DI COMUNITÀ,CASA DI COMUNITÀ,MANTOVA,MN
162,CASA DI COMUNITÀ FARINI MILANO,ASST FATEBENEFRATELLI SACCO,"VIA FARINI, 9 - MILANO (MI)",NaN,NaN,NaN,NaN,NaN,NaN,CASA DI COMUNITÀ,CASA DI COMUNITÀ,MILANO,MI
88,OSPEDALE C. CANTÙ,ASST OVEST MILANESE,"PIAZZA MUSSI, 1 - ABBIATEGRASSO (MI)",3.0,6.0,9.0,"[ENDOCRINOLOGIA E DIABETOLOGIA, GINECOLOGIA, M...","{""Da consigliare a parenti e amici"": 2.8, ""Pul...","{""Oculistica"": 1.0, ""Pronto soccorso"": 5.0, ""P...",OSPEDALE,SPOKE,ABBIATEGRASSO,MI
178,CASA DI COMUNITÀ UMBERTO I BERZO INFERIORE,ASST DELLA VALCAMONICA,"PIAZZA UMBERTO I, 7 - BERZO INFERIORE (BS)",NaN,NaN,NaN,NaN,NaN,NaN,CASA DI COMUNITÀ,CASA DI COMUNITÀ,BERZO INFERIORE,BS
144,CASA DI COMUNITÀ S. FRANCESCO PIOLTELLO,ASST MELEGNANO E DELLA MARTESANA,"VIA S. FRANCESCO, 16 - PIOLTELLO (MI)",NaN,NaN,NaN,NaN,NaN,NaN,CASA DI COMUNITÀ,CASA DI COMUNITÀ,PIOLTELLO,MI


In [48]:
def _parse_indirizzo(indirizzo: str) -> dict:
    if not indirizzo or pd.isna(indirizzo):
        return {}
    m = re.match(r'^(.+?)\s*-\s*(.+?)\s*\(([A-Z]{2})\)\s*$', indirizzo.strip())
    if m:
        street = re.sub(r',', '', m.group(1)).strip()
        city   = m.group(2).strip().title()
        prov   = m.group(3).upper()
        return {'street': street, 'city': city, 'prov': prov}
    return {}


def geocode(indirizzo: str, nome: str = None) -> tuple[float, float] | None:
    if not indirizzo or pd.isna(indirizzo):
        return None

    key = str(indirizzo).strip()
    parsed = _parse_indirizzo(indirizzo)
    BASE = 'https://nominatim.openstreetmap.org/search'
    HEADERS = {'User-Agent': 'auramed-geocoder/2.0'}

    def _get(params: dict) -> tuple[float, float] | None:
        time.sleep(2)
        try:
            resp = requests.get(BASE, params=params, headers=HEADERS, timeout=10)
            resp.raise_for_status()
            results = resp.json()
            if results:
                return float(results[0]['lat']), float(results[0]['lon'])
        except Exception as e:
            print(f'Errore su "{key}": {e}')
        return None

    # Tentativo 1: strutturato (via + città)
    coords = _get({
        'street': parsed['street'],
        'city': parsed['city'],
        'countrycodes': 'it',
        'format': 'json',
        'limit': 1,
    })

    if coords:
        return coords

    coords = _get({'q': f"{indirizzo}, Italia",
                   'countrycodes': 'it',
                   'format': 'json',
                   'limit': 1})
    if coords:
        return coords

    coords = _get({'q': f"{nome}, {parsed['city']}, Italia",
                   'countrycodes': 'it',
                   'format': 'json',
                   'limit': 1})
    if coords:
        return coords


# Test
test_indirizzi = [
    'VIA BORDINI, 9 - ANGERA (VA)',
    'VIA PALESTRO, 22 - MILANO (MI)',
    'VIA GIOVANNI PASCOLI, 36 - VIMERCATE (MB)',
]
for t in test_indirizzi:
    print(f'{t}  ->  {geocode(t)}')

VIA BORDINI, 9 - ANGERA (VA)  ->  (45.7755978, 8.579008)
VIA PALESTRO, 22 - MILANO (MI)  ->  (45.4721695, 9.2015853)
VIA GIOVANNI PASCOLI, 36 - VIMERCATE (MB)  ->  (45.634475, 9.3477305)


In [49]:
latitudini  = []
longitudini = []
totale = len(df_merged)

for i, row in tqdm(df_merged.iterrows(), total=totale):
    coords = geocode(row['INDIRIZZO'], nome=row['NOME_STRUTTURA'])
    latitudini.append(coords[0] if coords else None)
    longitudini.append(coords[1] if coords else None)

df_merged['LATITUDINE']  = latitudini
df_merged['LONGITUDINE'] = longitudini

non_geo = df_merged['LATITUDINE'].isna().sum()
print(f'\nCompletato. Non trovati: {non_geo}/{totale}')

if non_geo > 0:
    print('\nIndirizzi non geocodificati:')
    print(df_merged[df_merged['LATITUDINE'].isna()][['NOME_STRUTTURA', 'INDIRIZZO']].to_string())

df_merged[['NOME_STRUTTURA', 'INDIRIZZO', 'LATITUDINE', 'LONGITUDINE']].sample(5)


100%|██████████| 327/327 [16:31<00:00,  3.03s/it]


Completato. Non trovati: 17/327

Indirizzi non geocodificati:
                                         NOME_STRUTTURA                                                 INDIRIZZO
107                       IRCCS MAUGERI CASTEL GOFFREDO                  VIA OSPEDALE, 36 -  CASTEL GOFFREDO (MN)
108                            IRCCS MAUGERI MONTESCANO                  VIA PER MONTESCANO, 35 - MONTESCANO (PV)
127        OSPEDALE VILLA DEI COLLI DI LONATO DEL GARDA               VIA ARRIGA ALTA, 11 - LONATO DEL GARDA (BS)
151                 CASA DI COMUNITÀ OSPEDALE CALCINATE                       PIAZZA OSPEDALE, 3 - CALCINATE (BG)
159    CASA DI COMUNITÀ QUINTINI DI VONA CASSANO D'ADDA                  VIA Q. DI VONA, 41 - CASSANO D'ADDA (MI)
186  CASA DI COMUNITÀ CARDINAL DELL'ACQUA SESTO CALENDE        LARGO CARDINALE DELL'ACQUA, 1 - SESTO CALENDE (VA)
193       CASA DI COMUNITÀ VANONCINI SANT'OMOBONO TERME  VIA GIANANTONIO VANONCINI, 25 -  SANT'OMOBONO TERME (BG)
206           CASA DI COM

,NOME_STRUTTURA,INDIRIZZO,LATITUDINE,LONGITUDINE
318,CASA DI COMUNITÀ GIUSEPPE VERDI PONTE LAMBRO,"VIA GIUSEPPE VERDI, 3 - PONTE LAMBRO (CO)",NaN,NaN
62,IRCCS POLICLINICO SAN DONATO DI SAN DONATO MIL...,PIAZZA EDMONDO MALAN - SAN DONATO MILANESE (MI),45.410955,9.277692
20,OSPEDALE DI EDOLO,"PIAZZA DONATORI DI SANGUE, 1 - EDOLO (BS)",46.178756,10.331187
142,CASA DI COMUNITÀ MAGGIORE MARTINENGO,"PIAZZA MAGGIORE, 11 - MARTINENGO (BG)",45.570692,9.767320
89,OSPEDALE CIVILE DI LEGNANO,VIA PAPA GIOVANNI PAOLO II - LEGNANO (MI),45.585922,8.886345


In [50]:
mask = df_merged["NOME_STRUTTURA"] == "IRCCS MAUGERI CASTEL GOFFREDO"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.3013083,10.4785441]

mask = df_merged["NOME_STRUTTURA"] == "IRCCS MAUGERI MONTESCANO"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.0325518,9.2818479]

mask = df_merged["NOME_STRUTTURA"] == "OSPEDALE VILLA DEI COLLI DI LONATO DEL GARDA"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.4571342,10.4958144]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ OSPEDALE CALCINATE"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.6198516,9.7953154]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ QUINTINI DI VONA CASSANO D'ADDA"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.5228557,9.5132284]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ CARDINAL DELL'ACQUA SESTO CALENDE"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.7303017,8.6298946]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ VANONCINI SANT'OMOBONO TERME"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.8133656,9.5300692]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ CORTE DEI FRATI BELLUSCO"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.6163882,9.4147176]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ LUIGI CADORNA SUZZARA"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [44.9888422,10.74195]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ DONATORI DEL SANGUE LENO"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.3654434,10.2148178]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ DON MINZONI CASTELLANZA"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.6099961,8.8867093]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ GRAMSCI FAGNANO OLONA"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.6705114,8.8683928]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ BARBOLINI DARFO BOARIO TERME"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.875797,10.1789621]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ MONTEGRAPPA VIGEVANO"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.3219261,8.8450556]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ DONATORI EDOLO"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [46.1789936,10.3291521]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ GIUSEPPE VERDI PONTE LAMBRO"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.8179673,9.2209813]

mask = df_merged["NOME_STRUTTURA"] == "CASA DI COMUNITÀ GEN. REVERBERI VESTONE"
df_merged.loc[mask, ["LATITUDINE", "LONGITUDINE"]] = [45.7043645,10.3857049]

In [51]:
OUTPUT_FILE = "strutture_merged.csv"
df_merged.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

In [52]:
def build_aree(aree_str, rating_per_area):
    if isinstance(aree_str, list):
        aree = aree_str
    elif isinstance(aree_str, str):
        aree = ast.literal_eval(aree_str)
    else:
        aree = []  # NaN o None

    rating_map = {k.upper(): v for k, v in rating_per_area.items()}
    
    result = []
    for area in aree:
        entry = {"Nome": area}
        if area.upper() in rating_map:
            entry["Rating"] = rating_map[area.upper()]
        result.append(entry)
    return result

def row_to_doc(row):
    rating_per_area = json.loads(row["RATING_PER_AREA_JSON"]) if pd.notna(row["RATING_PER_AREA_JSON"]) else {}
    rating_dim = json.loads(row["RATING_DIMENSIONI_JSON"]) if pd.notna(row["RATING_DIMENSIONI_JSON"]) else {}

    return {
        "Nome": row["NOME_STRUTTURA"],
        "Città": row["COMUNE"],
        "Provincia": row["PROVINCIA"],
        "Indirizzo": row["INDIRIZZO"],
        "Ente": row["ASST"],
        "Classificazione": row["CLASSIFICAZIONE"],
        "Aree_specialistiche": build_aree(row["AREE_SPECIALISTICHE"], rating_per_area),
        "Rating_globale": row["RATING_GLOBALE"],
        "N_valutazioni": int(row["N_VALUTAZIONI"]) if pd.notna(row["N_VALUTAZIONI"]) else None,
        "N_aree": int(row["N_AREE"]) if pd.notna(row["N_AREE"]) else None,
        "Rating_dimensioni": rating_dim,
        "Latitudine": row["LATITUDINE"],
        "Longitudine": row["LONGITUDINE"],
    }

print(df_merged.index.is_unique)

docs = [row_to_doc(row) for _, row in df_merged.iterrows()]

with open("strutture_merged.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)

True
